# 05 Caching and Idempotency Keys (LiteLLM, 2026)

## What This Lesson Is
Prevent duplicate work and side effects through deterministic cache keys and idempotency semantics.

## Scientific Lens
- Concept: Idempotent request processing
- Measure: Cache hit ratio and duplicate-side-effect avoidance
- Validity Limit: In-memory caches are insufficient for distributed production systems.


## How It Works
1. Create deterministic request fingerprints.
2. Simulate idempotent replay.
3. Wrap live calls in cache semantics.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Idempotency demo uses SHA256 request fingerprinting")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import hashlib
import json

cache = {}
request = {
    "model": "openai/gpt-4.1-mini",
    "messages": [{"role": "user", "content": "Summarize patch notes"}],
    "idempotency_key": "release-2026-02-19",
}

fingerprint = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
cache.setdefault(fingerprint, {"status": "computed", "value": "summary-v1"})
cache.setdefault(fingerprint, {"status": "should-not-overwrite", "value": "summary-v2"})

print(cache[fingerprint])
assert cache[fingerprint]["value"] == "summary-v1"


In [ ]:
# Live Demo
import hashlib
import json
import os

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live cache demo: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live cache demo: OPENAI_API_KEY not set.")
    else:
        memo = {}
        req = {
            "model": "openai/gpt-4.1-mini",
            "prompt": "One sentence: define idempotency for API workflows.",
        }
        key = hashlib.sha256(json.dumps(req, sort_keys=True).encode()).hexdigest()
        if key not in memo:
            r = completion(
                model=req["model"],
                messages=[{"role": "user", "content": req["prompt"]}],
                api_key=api_key,
                timeout=20,
            )
            memo[key] = r.choices[0].message.content.strip()
            print("cache miss -> computed")
        else:
            print("cache hit")
        print(memo[key])


## Applied Labs
1. Include request temperature in fingerprint and observe cache segmentation.
2. Implement TTL expiry and verify stale eviction behavior.
3. Capture hit ratio under mixed repeat/new prompt workloads.

## Validation Checklist
- Fingerprint includes all fields that affect output semantics.
- Replay with same idempotency key does not duplicate compute path.
- Cache logic can distinguish miss, hit, and stale states.

## Further Reading
- [IETF Idempotency-Key Header Draft](https://datatracker.ietf.org/doc/draft-ietf-httpapi-idempotency-key-header/)
- [LiteLLM Caching Docs](https://docs.litellm.ai/docs/proxy/caching)
- [Stripe Idempotent Requests](https://stripe.com/docs/idempotency)
